# Journey into Sound  

Here we will look into the applications of deeplearning in the domain of audio. We will learn how to work with torchaudio library.


## ESC-50 dataset  

We will work with ESC-50 dataset which is a popular audio dataset with labeled collection of 2000 environmental audio recordings organized in 50 classes.  

You can access the dataset using this link : [click here](https://github.com/karolpiczak/ESC-50)  

Let's have a look at our dataset first.

In [1]:
import glob
from collections import Counter

esc50_list = [f.split('-')[-1].replace('wav',"")
              for f in glob.glob("./datasets/ESC-50/audio/*.wav")
              ]

Counter(esc50_list)

Counter()

In [2]:
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset    

class ESC50(Dataset):

    def __init__(self, path):
        files=Path(path).glob('*.wav')
        self.items = [(f,int(f.name.split('-')[-1].replace('.wav','')))
                      for f in files]
        
        self.length = len(self.items)

    def __getitem__(self, index):
        filename, label = self.items[index]
        audio_tensor, sample_rate = torchaudio.load(filename) 
        return audio_tensor,label
    
    def __len__(self):
        return self.length

In [3]:
import os
import shutil

# --- Configuration ---
# The folder where all your original .wav files are located.
# Change 'audio' to your folder's name if it's different.
SOURCE_DIR = '/kaggle/input/esc50/ESC-50-master/audio'

# The base folder where 'train', 'valid', and 'test' folders will be created.
# You can change 'data' to any name you prefer.
DEST_BASE_DIR = '/kaggle/working/esc50'
# --------------------

def organize_esc50_dataset(source_folder, dest_base_folder):
    """
    Organizes ESC-50 audio files from a single source folder into
    train, valid, and test subdirectories based on their fold prefix.
    """
    # Define destination paths
    train_dir = os.path.join(dest_base_folder, 'train')
    valid_dir = os.path.join(dest_base_folder, 'valid')
    test_dir = os.path.join(dest_base_folder, 'test')

    # 1. Create the destination directories if they don't exist
    print(f"Creating directories in '{dest_base_folder}'...")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(valid_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    print("Directories created successfully. ✅")

    # 2. Check if the source directory exists
    if not os.path.isdir(source_folder):
        print(f"❌ Error: Source directory '{source_folder}' not found.")
        print("Please make sure the script is in the correct location and the folder name is correct.")
        return

    # 3. Get a list of all .wav files
    all_files = [f for f in os.listdir(source_folder) if f.endswith('.wav')]
    if not all_files:
        print(f"⚠️ No .wav files found in '{source_folder}'.")
        return

    # 4. Iterate over files and move them
    print(f"\nMoving {len(all_files)} files...")
    moved_count = 0
    for filename in all_files:
        src_path = os.path.join(source_folder, filename)

        if filename.startswith(('1-', '2-', '3-')):
            dest_path = os.path.join(train_dir, filename)
            shutil.copy(src_path, dest_path)
            moved_count += 1
        elif filename.startswith('4-'):
            dest_path = os.path.join(valid_dir, filename)
            shutil.copy(src_path, dest_path)
            moved_count += 1
        elif filename.startswith('5-'):
            dest_path = os.path.join(test_dir, filename)
            shutil.copy(src_path, dest_path)
            moved_count += 1

    print(f"\n🎉 Process complete! Moved {moved_count} files successfully.")



organize_esc50_dataset(SOURCE_DIR, DEST_BASE_DIR)

Creating directories in '/kaggle/working/esc50'...
Directories created successfully. ✅

Moving 2000 files...

🎉 Process complete! Moved 2000 files successfully.


In [4]:
test_esc50 = ESC50('/kaggle/working/esc50/test')
train_esc50 = ESC50('/kaggle/working/esc50/train')
valid_esc50 = ESC50('/kaggle/working/esc50/valid')



In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_esc50, batch_size=64, shuffle=True)
test_loader = DataLoader(test_esc50, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_esc50, batch_size=64, shuffle=True)



Since we are done with loading the dataset. Let's design out Neural Network for the classification

In [7]:
import torch.nn as nn

class AudioNet(nn.Module):
    def __init__(self):
        super(AudioNet,self).__init__()
        self.conv1 = nn.Conv1d(1, 128, 5, 4)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(7)
        self.conv2 = nn.Conv1d(128, 128, 3)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(7)
        self.conv3 = nn.Conv1d(128, 256, 3)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool3 = nn.MaxPool1d(6)
        self.conv4 = nn.Conv1d(256, 512, 3)
        self.bn4 = nn.BatchNorm1d(512)
        self.pool4 = nn.MaxPool1d(6)
        self.avgpool = nn.AvgPool1d(30)
        self.fc1 = nn.Linear(512,50)

    def forward(self,x):
        x = self.conv1(x)
        x = nn.functional.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = nn.functional.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = nn.functional.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = nn.functional.relu(self.bn4(x))
        x = self.pool4(x)
        x = self.avgpool(x)
        x = x.permute(0,2,1)
        x = self.fc1(x)
        return nn.functional.log_softmax(x, dim=2).squeeze(1)

Now let's initialize our optimizer and loss function and get started.  
We will use Adam optimizer as usual and Cross Entropy loss for the loss function as this is a multi-class classification task.

In [8]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [9]:
import torchinfo

audio_net = AudioNet()

torchinfo.summary(audio_net)

Layer (type:depth-idx)                   Param #
AudioNet                                 --
├─Conv1d: 1-1                            768
├─BatchNorm1d: 1-2                       256
├─MaxPool1d: 1-3                         --
├─Conv1d: 1-4                            49,280
├─BatchNorm1d: 1-5                       256
├─MaxPool1d: 1-6                         --
├─Conv1d: 1-7                            98,560
├─BatchNorm1d: 1-8                       512
├─MaxPool1d: 1-9                         --
├─Conv1d: 1-10                           393,728
├─BatchNorm1d: 1-11                      1,024
├─MaxPool1d: 1-12                        --
├─AvgPool1d: 1-13                        --
├─Linear: 1-14                           25,650
Total params: 570,034
Trainable params: 570,034
Non-trainable params: 0

In [10]:
import torch.optim as optim

audio_net.to(device)
#torch.save(audio_net.state_dict(),'models/audionet.pth')

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(audio_net.parameters(), lr=1e-4, weight_decay=1e-4)
print(device)

cuda


Let's use the find_lr function from our learning_rate notebook to find figure out a better lr.

In [ ]:
import math

def find_lr(model, loss_fn, optimizer, init_val=1e-8, final_val=1):
    number_in_epoch = len(train_loader)-1
    update_step = (final_val / init_val)**(1/number_in_epoch)
    lr = init_val
    optimizer.param_groups[0]["lr"] = lr
    best_loss = 0.0
    batch_num = 0
    losses = []
    log_lrs = []

    for data in train_loader:
        batch_num+=1
        inputs,labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)        
        optimizer.zero_grad()
        outputs = model(inputs)

        loss = loss_fn(outputs,labels)

        if batch_num > 1 and loss > 4*best_loss:
            return log_lrs[10:-5], losses[10:-5]

        # Record best loss

        if loss < best_loss or batch_num == 1:
            best_loss = loss

        x = loss.item()
        losses.append(x)
        log_lrs.append(math.log10(lr))

        lr*=update_step
        optimizer.param_groups[0]['lr'] = lr
    
    return log_lrs[10:-5], losses[10:-5]

In [ ]:
import matplotlib.pyplot as plt


log,losses = find_lr(audio_net,loss_fn,optim)

beta = 0.95
avg_loss = 0
smoothed_losses = []
for i, loss in enumerate(losses):
    avg_loss = beta * avg_loss + (1 - beta) * loss
    # Debias the average to account for the initial cold start
    smoothed_losses.append(avg_loss / (1 - beta**(i + 1)))

# --- Plotting ---
fig = plt.figure(figsize=(10, 5))

# Plot the original noisy data (optional, for comparison)
plt.plot(log, losses, alpha=0.3, label='Original Losses')

# Plot the smoothed data
plt.plot(log, smoothed_losses, label='Smoothed Losses')

plt.xlabel("Log Learning Rate")
plt.ylabel("Loss")
plt.legend()
plt.show()

Now let's train the model

In [11]:
from tqdm import tqdm


def train_model(model,optimizer,loss_fn,train_loader,val_loader,epochs=20,device='cpu'):
    best_val_loss = float('inf')
    best_model=None
    for epoch in range(epochs):
        train_loss = 0.0
        val_loss = 0.0
        model.train()
        for batch in tqdm(train_loader):
            optimizer.zero_grad()
            inputs,target=batch
            inputs=inputs.to(device)
            target=target.to(device)
            output = model(inputs)
            loss = loss_fn(output,target)
            loss.backward()
            optimizer.step()
            train_loss+=loss.data.item()*inputs.size(0)
        train_loss/=len(train_loader.dataset)

        model.eval()
        num_correct=0
        num_examples=0
        with torch.no_grad():
            for batch in tqdm(val_loader):
                inputs,target=batch
                inputs=inputs.to(device)
                target=target.to(device)
                output = model (inputs)
                loss = loss_fn (output,target)
                val_loss+=loss.data.item()*inputs.size(0)
                predicted = torch.argmax(output, dim=1)
                num_correct += (predicted == target).sum().item()
                num_examples += len(target)
        val_loss/=len(val_loader.dataset)
        accuracy = num_correct/num_examples


        print('Epoch: {} , Training loss: {:.2f}, Validation loss: {:.2f}, accuracy: {:.2f}'.format(epoch,train_loss,val_loss,accuracy))

train_model(audio_net,optimizer,loss_fn,train_loader,valid_loader,epochs=50,device='cuda')

100%|██████████| 7/7 [00:05<00:00,  1.38it/s]


Epoch: 0 , Training loss: 3.75, Validation loss: 3.92, accuracy: 0.02


100%|██████████| 7/7 [00:05<00:00,  1.40it/s]


Epoch: 1 , Training loss: 3.40, Validation loss: 3.77, accuracy: 0.05


100%|██████████| 7/7 [00:04<00:00,  1.41it/s]


...

Epoch: 47 , Training loss: 1.49, Validation loss: 1.99, accuracy: 0.51


100%|██████████| 7/7 [00:04<00:00,  1.41it/s]


Epoch: 48 , Training loss: 1.49, Validation loss: 1.97, accuracy: 0.52


100%|██████████| 7/7 [00:05<00:00,  1.39it/s]


Epoch: 49 , Training loss: 1.47, Validation loss: 1.98, accuracy: 0.51
